# Dealer Gamma 03 — regime: does dealer gamma predict next-session volatility and autocorrelation?

Pre-registration: `discussions/Dealer_Gamma/001_claude_branch_preregistration.md`. Code: `src/data_engineering/modeling/dealer_gamma.py`.

Session d = (08:00 d, 08:00 d+1]. Predictor: $x = G_{08:05}/\text{ADV}_{30}$. Targets: log realized vol and lag-1 autocorrelation of 5-minute perpetual returns. Controls: log DVOL, lagged log RV, weekday. HAC SE (5 lags); coefficient per 1 SD of $x$.

**H:** β < 0 for both targets (long gamma → calmer, more reversion). PASS = one-sided p < 0.05 (discovery) / 0.10 (holdout) in both BTC and ETH. Holdout runs only with `DG_HOLDOUT=1`.

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import dealer_gamma as dg

ENV = dg.notebook_env('03', PROJECT_ROOT)
OUT, CACHE, HIST = ENV['out'], ENV['cache'], ENV['history']
PERIODS = ['discovery'] + (['holdout'] if ENV['holdout'] else [])
print(dg.VERSION, '| synthetic' if ENV['synthetic'] else '', '| periods', PERIODS)
print('data', ENV['data_root']); print('cache', CACHE); print('out', OUT)

In [ ]:
PAN, FITS, TER = {}, [], []
for c in dg.CURRENCIES:
    d = dg.load_cache(CACHE, c)
    PAN[c] = dg.daily_panel(d['grid'], dg.load_dvol(ENV['data_root'], c)).assign(currency=c)
    for per in PERIODS:
        for v in dg.VARIANTS + ('recon_1d',):
            FITS += [dict(r, currency=c) for r in dg.regime_fit(PAN[c], per, v)]
        TER.append(dg.terciles(PAN[c], per).assign(currency=c, period=per))
FITS = pd.DataFrame(FITS); TER = pd.concat(TER)
pd.concat(PAN.values()).to_csv(OUT / '03_panel.csv', index=False, float_format='%.6g')
FITS.to_csv(OUT / '03_fits.csv', index=False); TER.to_csv(OUT / '03_terciles.csv', index=False)
print(pd.concat(PAN.values()).groupby(['currency', 'period']).size())
display(FITS[FITS.variant == dg.PRIMARY].round(4)); display(TER.round(4))

In [ ]:
display(FITS.pivot_table(index=['period', 'variant'], columns=['currency', 'target'], values='t').round(2))
dec = dg.decide(FITS[FITS.variant == dg.PRIMARY].dropna(subset=['p_one_sided']), ['variant', 'target'])
dg.dump({'registration': dg.registration(), 'decision_primary': dec,
         'variant_sign_flags': FITS.assign(neg=FITS.beta_per_sd < 0).groupby(['period', 'target', 'currency']).neg.agg(['sum', 'size']).reset_index().to_dict('records')},
        OUT / '03_decision.json')
print(json.dumps(dec, indent=2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for j, (col, lab) in enumerate((('rv_day_pct', 'mean session realized vol, %'), ('ac1', 'mean AC1 of 5-min returns'))):
    t = TER[TER.period == 'discovery']
    for k, c in enumerate(dg.CURRENCIES):
        z = t[t.currency == c]
        ax[j].bar(np.arange(len(z)) + 0.4 * k, z[col], width=0.4, label=c)
    ax[j].set_xticks(np.arange(3) + 0.2, ['low (short γ)', 'mid', 'high (long γ)']); ax[j].set_title(lab); ax[j].legend()
plt.suptitle('Discovery: sessions by tercile of dealer gamma at 08:05'); plt.tight_layout()
plt.savefig(OUT / '03_terciles.png', dpi=110); plt.show()